In [2]:
import os
import glob
import xarray as xr

base_path = r"C:\Users\acann\OneDrive\Desktop\mtech major project\Anwesha Chakraborty\data\10 years data"   # change this

nc_files = glob.glob(
    os.path.join(base_path, "**", "*.nc"),
    recursive=True
)

print("Total NC files:", len(nc_files))

for file in nc_files:
    print("\n" + "=" * 100)
    print("FILE:", file)

    try:
        ds = xr.open_dataset(file)

        print("\nDimensions:")
        print(ds.dims)

        print("\nCoordinates:")
        print(list(ds.coords))

        print("\nData variables:")
        print(list(ds.data_vars))

        print("\nTime:")
        if "time" in ds.coords:
            print(ds["time"].min().values, "→", ds["time"].max().values)
        elif "valid_time" in ds.coords:
            print(ds["valid_time"].min().values, "→", ds["valid_time"].max().values)
        else:
            print("No standard time coordinate found")

        print("\nVariable details:")
        for var in ds.data_vars:
            print(
                f"  {var}: "
                f"shape={ds[var].shape}, "
                f"dtype={ds[var].dtype}, "
                f"units={ds[var].attrs.get('units', 'N/A')}"
            )

        ds.close()

    except Exception as e:
        print("ERROR:", e)

Total NC files: 20

FILE: C:\Users\acann\OneDrive\Desktop\mtech major project\Anwesha Chakraborty\data\10 years data\1202c0c9c0b25ccabd6b384e942b41ec\data_stream-oper_stepType-instant.nc

Dimensions:
FrozenMappingWarningOnValuesAccess({'valid_time': 7320, 'latitude': 5, 'longitude': 5})

Coordinates:
['number', 'valid_time', 'latitude', 'longitude', 'expver']

Data variables:
['t2m', 'blh']

Time:
2019-11-01T00:00:00.000000000 → 2023-12-31T23:00:00.000000000

Variable details:
  t2m: shape=(7320, 5, 5), dtype=float32, units=K
  blh: shape=(7320, 5, 5), dtype=float32, units=m

FILE: C:\Users\acann\OneDrive\Desktop\mtech major project\Anwesha Chakraborty\data\10 years data\2098890416a61e246dd3ac85b12cae2d\data_stream-oper_stepType-instant.nc

Dimensions:
FrozenMappingWarningOnValuesAccess({'valid_time': 17664, 'latitude': 5, 'longitude': 5})

Coordinates:
['number', 'valid_time', 'latitude', 'longitude', 'expver']

Data variables:
['d2m']

Time:
2016-07-01T00:00:00.000000000 → 2019-12-31

In [3]:
import pandas as pd
import numpy as np

In [4]:
base_path = r"C:\Users\acann\OneDrive\Desktop\mtech major project\Anwesha Chakraborty\data\10 years data"

output_file = os.path.join(base_path, "MASTER_NC_2016_2023.csv")

In [5]:
nc_files = glob.glob(
    os.path.join(base_path, "**", "*.nc"),
    recursive=True
)

print("Total NC files found:", len(nc_files))

if len(nc_files) == 0:
    raise FileNotFoundError("No .nc files found.")

Total NC files found: 20


In [6]:
wanted_variables = [
    "t2m",
    "d2m",
    "blh",
    "u10",
    "v10"
]

In [8]:
dataframes = []

for i, file in enumerate(nc_files, 1):

    print("\n" + "=" * 80)
    print(f"Processing {i}/{len(nc_files)}")
    print(os.path.basename(file))

    try:

        ds = xr.open_dataset(file)
        if "valid_time" in ds.coords:
            time_coord = "valid_time"

        elif "time" in ds.coords:
            time_coord = "time"

        else:
            print("WARNING: No time coordinate found. Skipping.")
            ds.close()
            continue
        variables = [
            var for var in wanted_variables
            if var in ds.data_vars
        ]

        if not variables:
            print("WARNING: No wanted variables found. Skipping.")
            ds.close()
            continue
        temp_df = ds[variables].mean(
            dim=["latitude", "longitude"],
            skipna=True
        ).to_dataframe().reset_index()

        ds.close()
        temp_df = temp_df[
            [time_coord] + variables
        ]

        # Rename time
        temp_df = temp_df.rename(
            columns={time_coord: "Datetime"}
        )

        temp_df["Datetime"] = pd.to_datetime(
            temp_df["Datetime"]
        )

        if "t2m" in temp_df.columns:
            temp_df["t2m"] = temp_df["t2m"] - 273.15

        if "d2m" in temp_df.columns:
            temp_df["d2m"] = temp_df["d2m"] - 273.15

        temp_df = (
            temp_df
            .groupby("Datetime", as_index=False)
            .mean(numeric_only=True)
        )

        dataframes.append(temp_df)

        print(
            "Variables:",
            variables
        )

        print(
            "Time:",
            temp_df["Datetime"].min(),
            "→",
            temp_df["Datetime"].max()
        )

        print(
            "Rows:",
            len(temp_df)
        )

    except Exception as e:

        print("ERROR:", e)


Processing 1/20
data_stream-oper_stepType-instant.nc
Variables: ['t2m', 'blh']
Time: 2019-11-01 00:00:00 → 2023-12-31 23:00:00
Rows: 7320

Processing 2/20
data_stream-oper_stepType-instant.nc
Variables: ['d2m']
Time: 2016-07-01 00:00:00 → 2019-12-31 23:00:00
Rows: 17664

Processing 3/20
data_stream-oper_stepType-instant.nc
Variables: ['t2m']
Time: 2019-06-01 00:00:00 → 2023-10-31 23:00:00
Rows: 18360

Processing 4/20
data_stream-oper_stepType-instant.nc
Variables: ['u10', 'v10']
Time: 2020-07-01 00:00:00 → 2023-09-30 23:00:00
Rows: 8832

Processing 5/20
data_stream-oper_stepType-instant.nc
Variables: ['u10', 'v10']
Time: 2020-04-01 00:00:00 → 2023-06-30 23:00:00
Rows: 8736

Processing 6/20
data_stream-oper_stepType-instant.nc
Variables: ['blh']
Time: 2019-01-01 00:00:00 → 2023-05-31 23:00:00
Rows: 18144

Processing 7/20
data_stream-oper_stepType-instant.nc
Variables: ['u10', 'v10']
Time: 2020-10-01 00:00:00 → 2023-12-31 23:00:00
Rows: 8832

Processing 8/20
data_stream-oper_stepType-in

In [9]:
print("\n" + "=" * 80)
print("COMBINING FILES")

if not dataframes:
    raise ValueError("No NC files were successfully processed.")

master = pd.concat(
    dataframes,
    ignore_index=True
)


COMBINING FILES


In [10]:
master = (
    master
    .groupby("Datetime", as_index=False)
    .mean(numeric_only=True)
)

In [11]:
master = master.sort_values(
    "Datetime"
).reset_index(drop=True)

In [12]:
master = master.drop_duplicates(
    subset=["Datetime"]
).reset_index(drop=True)


In [13]:
master.shape

(70128, 6)

In [14]:
for var in wanted_variables:

    if var in master.columns:

        valid = master[
            master[var].notna()
        ]

        if len(valid) > 0:

            print(
                f"\n{var}:"
            )

            print(
                "  Start:",
                valid["Datetime"].min()
            )

            print(
                "  End:",
                valid["Datetime"].max()
            )

            print(
                "  Valid rows:",
                len(valid)
            )



t2m:
  Start: 2016-01-01 00:00:00
  End: 2023-12-31 23:00:00
  Valid rows: 70128

d2m:
  Start: 2016-01-01 00:00:00
  End: 2023-12-31 23:00:00
  Valid rows: 70128

blh:
  Start: 2016-01-01 00:00:00
  End: 2023-12-31 23:00:00
  Valid rows: 70128

u10:
  Start: 2016-01-01 00:00:00
  End: 2023-12-31 23:00:00
  Valid rows: 70128

v10:
  Start: 2016-01-01 00:00:00
  End: 2023-12-31 23:00:00
  Valid rows: 70128


In [17]:
missing = master.isna().sum()
print(missing)

Datetime    0
t2m         0
blh         0
d2m         0
u10         0
v10         0
dtype: int64


In [20]:
master.to_csv(
    output_file,
    index=False
)

In [19]:
print(
    "Saved to:"
)

print(output_file)

Saved to:
C:\Users\acann\OneDrive\Desktop\mtech major project\Anwesha Chakraborty\data\10 years data\MASTER_NC_2016_2023.csv


In [22]:
import os
import glob
import xarray as xr
import pandas as pd
import numpy as np
base_path = r"C:\Users\acann\OneDrive\Desktop\mtech major project\Anwesha Chakraborty\data\10 years data"

output_file = os.path.join(
    base_path,
    "MASTER_NC_2016_2023.csv"
)
files = glob.glob(
    os.path.join(base_path, "**", "*.nc"),
    recursive=True
)

print("Total NC files:", len(files))
variable_data = {
    "t2m": [],
    "d2m": [],
    "u10": [],
    "v10": [],
    "blh": []
}

for file in files:

    print("\nProcessing:", os.path.basename(file))

    ds = xr.open_dataset(file)
    time = pd.to_datetime(
        ds["valid_time"].values
    )
    for var in variable_data.keys():

        if var in ds:

            # Average 5 × 5 spatial grid
            values = ds[var].mean(
                dim=["latitude", "longitude"],
                skipna=True
            ).values

            temp_df = pd.DataFrame({
                "Datetime": time,
                var: values
            })

            variable_data[var].append(
                temp_df
            )

    ds.close()
merged_variables = {}

for var, dfs in variable_data.items():

    print("\nMerging:", var)

    if len(dfs) == 0:

        print("  No data found.")

        continue

    temp = pd.concat(
        dfs,
        ignore_index=True
    )
    temp = (
        temp
        .groupby("Datetime", as_index=False)[var]
        .mean()
    )

    merged_variables[var] = temp

    print(
        "  Start:",
        temp["Datetime"].min()
    )

    print(
        "  End:",
        temp["Datetime"].max()
    )

    print(
        "  Rows:",
        len(temp)
    )

print("\n" + "=" * 80)
print("MERGING VARIABLES BY DATETIME")
print("=" * 80)

master = None

for var in [
    "t2m",
    "d2m",
    "u10",
    "v10",
    "blh"
]:

    if var not in merged_variables:
        continue

    if master is None:

        master = merged_variables[var]

    else:

        master = master.merge(
            merged_variables[var],
            on="Datetime",
            how="outer"
        )


master = master.sort_values(
    "Datetime"
).reset_index(drop=True)


if "t2m" in master.columns:

    master["Temperature"] = (
        master["t2m"] - 273.15
    )

if "d2m" in master.columns:

    dew = (
        master["d2m"] - 273.15
    )
if (
    "t2m" in master.columns
    and "d2m" in master.columns
):

    temp = (
        master["t2m"] - 273.15
    )

    dew = (
        master["d2m"] - 273.15
    )

    # Magnus formula
    saturation_vapor_pressure = np.exp(
        (17.625 * temp) /
        (243.04 + temp)
    )

    actual_vapor_pressure = np.exp(
        (17.625 * dew) /
        (243.04 + dew)
    )

    master["RH"] = (
        100 *
        actual_vapor_pressure /
        saturation_vapor_pressure
    )
    master["RH"] = master["RH"].clip(
        lower=0,
        upper=100
    )
if (
    "u10" in master.columns
    and "v10" in master.columns
):

    master["WindSpeed"] = np.sqrt(
        master["u10"]**2 +
        master["v10"]**2
    )

if "blh" in master.columns:

    master["BLH"] = master["blh"]

final_columns = [
    "Datetime",
    "Temperature",
    "RH",
    "WindSpeed",
    "BLH"
]

master = master[
    [
        col
        for col in final_columns
        if col in master.columns
    ]
]

master = master.sort_values(
    "Datetime"
).reset_index(drop=True)

print("\n" + "=" * 80)
print("FINAL MASTER NC DATASET")
print("=" * 80)

print("\nShape:")
print(master.shape)

print("\nColumns:")
print(master.columns.tolist())

print("\nTime coverage:")
print(master["Datetime"].min())
print("→")
print(master["Datetime"].max())

print("\nMissing values:")

print(
    master.isnull().sum()
)

print("\nMissing percentage:")

print(
    (master.isnull().mean() * 100).round(2)
)

print("\n" + "=" * 80)
print("VARIABLE RANGES")
print("=" * 80)

for col in master.columns:

    if col != "Datetime":

        print(f"\n{col}")

        print(
            "Min:",
            master[col].min()
        )

        print(
            "Max:",
            master[col].max()
        )

        print(
            "Mean:",
            master[col].mean()
        )
print("\n" + "=" * 80)

print(
    "Duplicate timestamps:",
    master["Datetime"].duplicated().sum()
)

time_diff = master["Datetime"].diff()

print("\nTime difference counts:")

print(
    time_diff.value_counts().head(10)
)
master.to_csv(
    output_file,
    index=False
)

print("\n" + "=" * 80)

print("Saved:")
print(output_file)

Total NC files: 20

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_stepType-instant.nc

Processing: data_stream-oper_s